# Contact Form Intake API

This lab deploys **API Management** and **Service Bus** to create a simple contact form intake API.

**Architecture:**

![Contact Form Architecture](../../images/contact-form.png)

**Features:**
- API Management BasicV2 tier (~5 min deployment)
- No API key required (educational purposes)
- Rate limiting at the edge (10 calls/min per IP)
- APIM authenticates to Service Bus using Managed Identity
- Messages queued for asynchronous processing

### 0 Initialize notebook variables

In [32]:
import os
import sys
import json
import datetime
import hashlib

# Add shared utilities
sys.path.insert(1, '../../shared')
import utils

# Configuration
deployment_name = "contact-form"
resource_group_location = "westus2"

# Get subscription ID for unique naming
subscription_id = utils.get_current_subscription()
unique_suffix = hashlib.md5(subscription_id.encode()).hexdigest()[:8] if subscription_id else "default"

# Resource names
resource_group_name = f"rg-{deployment_name}"

print("Configuration:")
print(f"  Resource Group: {resource_group_name}")
print(f"  Location: {resource_group_location}")

✅ Retrieved Azure account ⌚ 14:14:30
ℹ️  Using Subscription: ME-MngEnvMCAP198880-odaibert-1 (2557a18b-db35-40e8-9977-08845cf4192a)
Configuration:
  Resource Group: rg-contact-form
  Location: westus2
  Unique Suffix: beef5c0b


### 1 Verify Azure CLI and connected subscription

In [2]:
# Verify Azure CLI
output = utils.run("az account show", "Azure CLI is configured", "Azure CLI not configured - run 'az login'")

if output.success and output.json_data:
    print(f"Subscription: {output.json_data['name']}")
    print(f"Subscription ID: {output.json_data['id']}")
    print(f"Tenant ID: {output.json_data['tenantId']}")


✅ Azure CLI is configured ⌚ 11:37:23
Subscription: ME-MngEnvMCAP198880-odaibert-1
Subscription ID: 2557a18b-db35-40e8-9977-08845cf4192a
Tenant ID: 6f4e776f-28a2-4303-879c-d1dba3028420


### 2 Create Resource Group

In [3]:
# Create resource group
utils.create_resource_group(resource_group_name, resource_group_location)


✅ Created resource group 'rg-contact-form' in westus2 ⌚ 11:38:05


True

### 3 Deploy Infrastructure using Bicep

In [29]:
# Deploy infrastructure
outputs = utils.deploy_bicep(
    resource_group=resource_group_name,
    template_file="main.bicep",
    parameters={"location": resource_group_location}
)

if outputs:
    print("Deployed Resources:")
    for key, value in outputs.items():
        print(f"  {key}: {value['value'] if isinstance(value, dict) else value}")


ℹ️  Deploying main.bicep...
✅ Deployment completed ⌚ 14:14:16
Deployed Resources:
  apimContactApiPath: https://apim-gateway-learning-yxxehbk6crs7y.azure-api.net/contact/submit
  apimGatewayUrl: https://apim-gateway-learning-yxxehbk6crs7y.azure-api.net
  apimName: apim-gateway-learning-yxxehbk6crs7y
  serviceBusNamespaceName: sb-contact-yxxehbk6crs7y
  serviceBusQueueName: contact-intake
  serviceBusQueueUri: https://sb-contact-yxxehbk6crs7y.servicebus.windows.net/contact-intake


### 4 Get Deployment Outputs

In [33]:
# Get deployment outputs
outputs = utils.get_deployment_outputs(resource_group_name)

if outputs:
    apim_gateway_url = outputs.get('apimGatewayUrl', '')
    apim_name = outputs.get('apimName', '')
    service_bus_namespace_name = outputs.get('serviceBusNamespaceName', '')
    service_bus_queue_name = outputs.get('serviceBusQueueName', '')
    service_bus_queue_uri = outputs.get('serviceBusQueueUri', '')

    print(f"API Management Gateway: {apim_gateway_url}")
    print(f"Service Bus Namespace: {service_bus_namespace_name}")
    print(f"Service Bus Queue: {service_bus_queue_name}")
    print(f"Service Bus Queue URI: {service_bus_queue_uri}")
else:
    print("Failed to retrieve deployment outputs")


✅ Retrieved deployment outputs ⌚ 14:14:35
API Management Gateway: https://apim-gateway-learning-yxxehbk6crs7y.azure-api.net
Service Bus Namespace: sb-contact-yxxehbk6crs7y
Service Bus Queue: contact-intake
Service Bus Queue URI: https://sb-contact-yxxehbk6crs7y.servicebus.windows.net/contact-intake


### 5 Test the Contact API

In [34]:
import requests
import uuid

# Test payload
payload = {
    "id": f"contact-{uuid.uuid4().hex[:8]}",
    "name": "Jane Doe",
    "email": "jane@example.com",
    "subject": "Question about pricing",
    "message": "Can you share your pricing tiers?",
    "submittedAt": datetime.datetime.now(datetime.UTC).strftime("%Y-%m-%dT%H:%M:%SZ")
}

# API endpoint (no authentication required)
api_url = f"{apim_gateway_url}/contact/submit"
headers = {"Content-Type": "application/json"}

print(f"Sending request to: {api_url}")
print(f"Payload: {json.dumps(payload, indent=2)}")

# Make request
response = requests.post(api_url, json=payload, headers=headers)

print(f"\nResponse Status: {response.status_code}")
print(f"Response Body: {json.dumps(response.json(), indent=2)}")

/var/folders/4n/0_6blvq90b70kbdd2bfyz4y80000gn/T/ipykernel_92230/2617221317.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "submittedAt": datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")


Sending request to: https://apim-gateway-learning-yxxehbk6crs7y.azure-api.net/contact/submit
Payload: {
  "id": "contact-21e75708",
  "name": "Jane Doe",
  "email": "jane@example.com",
  "subject": "Question about pricing",
  "message": "Can you share your pricing tiers?",
  "submittedAt": "2026-02-04T19:14:38Z"
}

Response Status: 202
Response Body: {
  "status": "queued",
  "messageId": "5f508e54-2d06-421f-b982-15b52201631b"
}


### 6 Verify Queue Message Count

In [35]:
# Check the number of active messages in the queue
result = utils.run(
    f'az servicebus queue show --resource-group {resource_group_name} --namespace-name {service_bus_namespace_name} --name {service_bus_queue_name} --query "countDetails.activeMessageCount" -o tsv',
    "Retrieved queue message count",
    "Failed to get queue message count"
)

if result.success:
    print(f"Active messages in queue: {result.output}")

✅ Retrieved queue message count ⌚ 14:14:44
Active messages in queue: 5


### 7 Clean up resources

In [9]:
# Uncomment to delete resources
# utils.delete_resource_group(resource_group_name)
print(f"To delete resources, run: az group delete --name {resource_group_name} --yes")

To delete resources, run: az group delete --name rg-contact-form --yes
